# Feature Mapping and Validation Support

 The purpose of this work is to understand the current feature engineering pipeline, map raw input fields to engineered features and perform basic validation checks on the generated feature dataset.

The current raw input structure includes:
- timestamp
- location
- severity
- duration_hours
- cyber_incidents

These fields are used to generate hazard, cyber, temporal, geo, combined risk and anomaly-related features.

## Imports

In [ ]:
import pandas as pd
import numpy as np

## Sample Input Dataset

In [ ]:
df = pd.DataFrame({
    "timestamp": ["2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04"],
    "location": ["VIC", "VIC", "NSW", "VIC"],
    "severity": [3, 5, 8, 6],
    "duration_hours": [2, 4, 6, 3],
    "cyber_incidents": [10, 15, 30, 20]
})

df

,timestamp,location,severity,duration_hours,cyber_incidents
0,2026-01-01,VIC,3,2,10
1,2026-01-02,VIC,5,4,15
2,2026-01-03,NSW,8,6,30
3,2026-01-04,VIC,6,3,20


## Raw Data Understanding

The current feature engineering pipeline works with five main raw fields:
- `timestamp` for time-based sequencing and event timing
- `location` for geographical grouping and region-level features
- `severity` for hazard intensity and risk-related calculations
- `duration_hours` for event impact and intensity measures
- `cyber_incidents` for cyber activity, spikes, trends, and anomaly detection

In [ ]:
class FeatureEngineer:

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()


    # 1. HAZARD FEATURES

    def create_hazard_features(self):

        self.df['disaster_severity_score'] = self.df['severity'] * 1.5
        self.df['event_intensity_index'] = self.df['severity'] * self.df['duration_hours']
        self.df['hazard_normalized'] = self.df['severity'] / self.df['severity'].max()
        self.df['hazard_frequency'] = self.df['severity'].rolling(3).count()

        self.df['severity_change_rate'] = self.df['severity'].diff().fillna(0)
        self.df['hazard_trend_index'] = self.df['severity'].rolling(3).mean().fillna(0)


        self.df['severity_volatility'] = self.df['severity'].rolling(3).std().fillna(0)
        self.df['multi_event_overlap_flag'] = (self.df['severity'] > 7).astype(int)

        return self.df

    # 2. CYBER FEATURES

    def create_cyber_features(self):

        self.df['cyber_incident_count'] = self.df['cyber_incidents']
        self.df['cyber_intensity_score'] = self.df['cyber_incidents'] / self.df['cyber_incidents'].max()

        self.df['scam_spike_rate'] = self.df['cyber_incidents'].diff().fillna(0)
        self.df['cyber_attack_frequency'] = self.df['cyber_incidents'].rolling(3).count()

        self.df['cyber_growth_rate'] = self.df['cyber_incidents'].pct_change().fillna(0)


        self.df['phishing_activity_level'] = self.df['cyber_incidents'] * 0.6
        self.df['misinformation_volume'] = self.df['cyber_incidents'] * 0.4

        self.df['cyber_volatility'] = self.df['cyber_incidents'].rolling(3).std().fillna(0)
        self.df['attack_acceleration'] = self.df['cyber_incidents'].diff().diff().fillna(0)
        self.df['incident_peak_flag'] = (self.df['cyber_incidents'] > self.df['cyber_incidents'].mean()).astype(int)

        return self.df


    # 3. TEMPORAL FEATURES

    def create_temporal_features(self):

        self.df['timestamp'] = pd.to_datetime(self.df['timestamp'])
        self.df = self.df.sort_values('timestamp')

        self.df['rolling_cyber_mean'] = self.df['cyber_incidents'].rolling(3).mean().fillna(0)
        self.df['rolling_cyber_std'] = self.df['cyber_incidents'].rolling(3).std().fillna(0)

        self.df['cyber_spike'] = self.df['cyber_incidents'].diff().fillna(0)
        self.df['trend_slope_cyber_activity'] = self.df['cyber_incidents'].diff().fillna(0)

        self.df['time_since_last_event'] = self.df['timestamp'].diff().dt.total_seconds().fillna(0)


        self.df['moving_average_ratio'] = (
            self.df['cyber_incidents'] / self.df['rolling_cyber_mean'].replace(0, 1)
        )

        self.df['ema'] = self.df['cyber_incidents'].ewm(span=3).mean()
        self.df['lag_1'] = self.df['cyber_incidents'].shift(1).fillna(0)
        self.df['lag_2'] = self.df['cyber_incidents'].shift(2).fillna(0)

        self.df['time_decay_factor'] = np.exp(-0.1 * np.arange(len(self.df)))

        return self.df


    # 4. GEO FEATURES

    def create_geo_features(self):

        self.df['geo_risk_zone_score'] = np.where(
            self.df['severity'] > 7, 1,
            np.where(self.df['severity'] > 4, 0.5, 0.2)
        )

        self.df['high_risk_region_flag'] = (self.df['severity'] > 7).astype(int)
        self.df['incident_density_by_region'] = self.df['cyber_incidents'] / (self.df['duration_hours'] + 1)


        self.df['location_encoded'] = self.df['location'].astype('category').cat.codes


        self.df['regional_event_count'] = self.df.groupby('location')['severity'].transform('count')
        self.df['geo_cluster_id'] = self.df['location_encoded']

        return self.df


    # 5. COMBINED RISK FEATURES

    def create_risk_features(self):

        self.df['combined_risk_index'] = (
            self.df['disaster_severity_score'] *
            self.df['cyber_incident_count']
        )

        self.df['event_impact_multiplier'] = (
            self.df['event_intensity_index'] *
            self.df['cyber_intensity_score']
        )

        self.df['disaster_cyber_correlation_score'] = (
            self.df['severity'] * self.df['cyber_incidents']
        )

        self.df['anomaly_risk_signal'] = (
            self.df['cyber_spike'] *
            self.df['disaster_severity_score']
        )

        self.df['risk_trend_index'] = self.df['combined_risk_index'].rolling(3).mean().fillna(0)


        self.df['multi_factor_risk_score'] = (
            self.df['combined_risk_index'] +
            self.df['cyber_volatility'] +
            self.df['severity_volatility']
        )

        self.df['hazard_weighted_cyber_score'] = (
            self.df['severity'] * self.df['cyber_intensity_score']
        )

        self.df['adaptive_risk_index'] = (
            self.df['multi_factor_risk_score'] /
            (self.df['rolling_cyber_mean'] + 1)
        )

        return self.df


    # 6. ANOMALY FEATURES

    def create_anomaly_features(self):

        mean = self.df['cyber_incidents'].mean()
        std = self.df['cyber_incidents'].std() + 1e-5

        self.df['z_score_cyber_activity'] = (
            (self.df['cyber_incidents'] - mean) / std
        )

        self.df['deviation_from_mean'] = self.df['cyber_incidents'] - mean

        self.df['outlier_flag'] = (abs(self.df['z_score_cyber_activity']) > 2).astype(int)
        self.df['threshold_breach_flag'] = (self.df['cyber_incidents'] > mean).astype(int)

        return self.df


    # RUN ALL

    def run(self):

        self.create_hazard_features()
        self.create_cyber_features()
        self.create_temporal_features()
        self.create_geo_features()
        self.create_risk_features()
        self.create_anomaly_features()

        return self.df

## Run the Feature Engineering Pipeline

In [ ]:
fe = FeatureEngineer(df)
result = fe.run()

result.head()

,timestamp,location,severity,duration_hours,cyber_incidents,disaster_severity_score,event_intensity_index,hazard_normalized,hazard_frequency,severity_change_rate,...,disaster_cyber_correlation_score,anomaly_risk_signal,risk_trend_index,multi_factor_risk_score,hazard_weighted_cyber_score,adaptive_risk_index,z_score_cyber_activity,deviation_from_mean,outlier_flag,threshold_breach_flag
0,2026-01-01,VIC,3,2,10,4.5,6,0.375,NaN,0.0,...,30,0.0,0.0,45.000000,1.0,45.000000,-1.024694,-8.75,0,0
1,2026-01-02,VIC,5,4,15,7.5,20,0.625,NaN,2.0,...,75,37.5,0.0,112.500000,2.5,112.500000,-0.439155,-3.75,0,0
2,2026-01-03,NSW,8,6,30,12.0,48,1.000,3.0,3.0,...,240,180.0,172.5,372.924941,8.0,19.289221,1.317464,11.25,0,1
3,2026-01-04,VIC,6,3,20,9.0,18,0.750,3.0,-2.0,...,120,-90.0,217.5,189.165151,4.0,8.345521,0.146385,1.25,0,1


## Output Shape and Columns

In [ ]:
print("Feature dataset shape:", result.shape)

print("\nGenerated columns:")
for col in result.columns:
    print(col)

Feature dataset shape: (4, 51)

Generated columns:
timestamp
location
severity
duration_hours
cyber_incidents
disaster_severity_score
event_intensity_index
hazard_normalized
hazard_frequency
severity_change_rate
hazard_trend_index
severity_volatility
multi_event_overlap_flag
cyber_incident_count
cyber_intensity_score
scam_spike_rate
cyber_attack_frequency
cyber_growth_rate
phishing_activity_level
misinformation_volume
cyber_volatility
attack_acceleration
incident_peak_flag
rolling_cyber_mean
rolling_cyber_std
cyber_spike
trend_slope_cyber_activity
time_since_last_event
moving_average_ratio
ema
lag_1
lag_2
time_decay_factor
geo_risk_zone_score
high_risk_region_flag
incident_density_by_region
location_encoded
regional_event_count
geo_cluster_id
combined_risk_index
event_impact_multiplier
disaster_cyber_correlation_score
anomaly_risk_signal
risk_trend_index
multi_factor_risk_score
hazard_weighted_cyber_score
adaptive_risk_index
z_score_cyber_activity
deviation_from_mean
outlier_flag
thres

## Feature Mapping

This section maps the available raw input fields to the possible engineered features generated by the feature engineering pipeline.

## Raw Field to Feature Mapping Table

In [ ]:
mapping_data = {
    "Raw Field": [
        "severity", "severity", "severity", "severity", "severity", "severity", "severity", "severity", "severity",
        "duration_hours", "duration_hours",
        "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents",
        "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents",
        "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents",
        "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents", "cyber_incidents",
        "timestamp", "timestamp",
        "location", "location", "location"
    ],
    "Engineered Feature": [
        "disaster_severity_score", "hazard_normalized", "hazard_frequency", "severity_change_rate",
        "hazard_trend_index", "severity_volatility", "multi_event_overlap_flag",
        "geo_risk_zone_score", "high_risk_region_flag",
        "event_intensity_index", "incident_density_by_region",
        "cyber_incident_count", "cyber_intensity_score", "scam_spike_rate",
        "cyber_attack_frequency", "cyber_growth_rate", "phishing_activity_level",
        "misinformation_volume", "cyber_volatility", "attack_acceleration",
        "incident_peak_flag", "rolling_cyber_mean", "rolling_cyber_std", "cyber_spike",
        "trend_slope_cyber_activity", "moving_average_ratio", "ema", "lag_1", "lag_2",
        "z_score_cyber_activity", "threshold_breach_flag",
        "time_since_last_event", "time_decay_factor",
        "location_encoded", "regional_event_count", "geo_cluster_id"
    ]
}

mapping_df = pd.DataFrame(mapping_data)
mapping_df

,Raw Field,Engineered Feature
0,severity,disaster_severity_score
1,severity,hazard_normalized
2,severity,hazard_frequency
3,severity,severity_change_rate
4,severity,hazard_trend_index
5,severity,severity_volatility
6,severity,multi_event_overlap_flag
7,severity,geo_risk_zone_score
8,severity,high_risk_region_flag
9,duration_hours,event_intensity_index


## Missing Values Check

In [ ]:
print("Missing values in feature dataset:")
print(result.isna().sum())

Missing values in feature dataset:
timestamp                           0
location                            0
severity                            0
duration_hours                      0
cyber_incidents                     0
disaster_severity_score             0
event_intensity_index               0
hazard_normalized                   0
hazard_frequency                    2
severity_change_rate                0
hazard_trend_index                  0
severity_volatility                 0
multi_event_overlap_flag            0
cyber_incident_count                0
cyber_intensity_score               0
scam_spike_rate                     0
cyber_attack_frequency              2
cyber_growth_rate                   0
phishing_activity_level             0
misinformation_volume               0
cyber_volatility                    0
attack_acceleration                 0
incident_peak_flag                  0
rolling_cyber_mean                  0
rolling_cyber_std                   0
cyber_spike    

## Numeric Summary

In [ ]:
numeric_cols = result.select_dtypes(include=[np.number]).columns

print("Summary statistics for numeric features:")
print(result[numeric_cols].describe())

Summary statistics for numeric features:
       severity  duration_hours  cyber_incidents  disaster_severity_score  \
count  4.000000        4.000000         4.000000                 4.000000   
mean   5.500000        3.750000        18.750000                 8.250000   
std    2.081666        1.707825         8.539126                 3.122499   
min    3.000000        2.000000        10.000000                 4.500000   
25%    4.500000        2.750000        13.750000                 6.750000   
50%    5.500000        3.500000        17.500000                 8.250000   
75%    6.500000        4.500000        22.500000                 9.750000   
max    8.000000        6.000000        30.000000                12.000000   

       event_intensity_index  hazard_normalized  hazard_frequency  \
count               4.000000           4.000000               2.0   
mean               23.000000           0.687500               3.0   
std                17.776389           0.260208           

## Validation Checks

In [ ]:
validation_results = {
    "no_missing_values": result.isna().sum().sum() == 0,
    "severity_non_negative": (result["severity"] >= 0).all(),
    "cyber_incidents_non_negative": (result["cyber_incidents"] >= 0).all(),
    "combined_risk_index_exists": "combined_risk_index" in result.columns,
    "hazard_feature_exists": "disaster_severity_score" in result.columns,
    "temporal_feature_exists": "time_since_last_event" in result.columns,
    "geo_feature_exists": "geo_cluster_id" in result.columns,
    "anomaly_feature_exists": "z_score_cyber_activity" in result.columns
}
for check, status in validation_results.items():
    print(f"{check}: {'Passed' if status else 'Failed'}")

no_missing_values: Failed
severity_non_negative: Passed
cyber_incidents_non_negative: Passed
combined_risk_index_exists: Passed
hazard_feature_exists: Passed
temporal_feature_exists: Passed
geo_feature_exists: Passed
anomaly_feature_exists: Passed


## Feature Groups

In [ ]:
hazard_features = [
    "disaster_severity_score", "event_intensity_index", "hazard_normalized",
    "hazard_frequency", "severity_change_rate", "hazard_trend_index",
    "severity_volatility", "multi_event_overlap_flag"
]

cyber_features = [
    "cyber_incident_count", "cyber_intensity_score", "scam_spike_rate",
    "cyber_attack_frequency", "cyber_growth_rate", "phishing_activity_level",
    "misinformation_volume", "cyber_volatility", "attack_acceleration",
    "incident_peak_flag"
]

temporal_features = [
    "rolling_cyber_mean", "rolling_cyber_std", "cyber_spike",
    "trend_slope_cyber_activity", "time_since_last_event",
    "moving_average_ratio", "ema", "lag_1", "lag_2", "time_decay_factor"
]

geo_features = [
    "geo_risk_zone_score", "high_risk_region_flag", "incident_density_by_region",
    "location_encoded", "regional_event_count", "geo_cluster_id"
]

risk_features = [
    "combined_risk_index", "event_impact_multiplier",
    "disaster_cyber_correlation_score", "anomaly_risk_signal",
    "risk_trend_index", "multi_factor_risk_score",
    "hazard_weighted_cyber_score", "adaptive_risk_index"
]

anomaly_features = [
    "z_score_cyber_activity", "deviation_from_mean",
    "outlier_flag", "threshold_breach_flag"
]

print("Hazard Features:")
print(hazard_features)

print("\nCyber Features:")
print(cyber_features)

print("\nTemporal Features:")
print(temporal_features)

print("\nGeo Features:")
print(geo_features)

print("\nRisk Features:")
print(risk_features)

print("\nAnomaly Features:")
print(anomaly_features)

Hazard Features:
['disaster_severity_score', 'event_intensity_index', 'hazard_normalized', 'hazard_frequency', 'severity_change_rate', 'hazard_trend_index', 'severity_volatility', 'multi_event_overlap_flag']

Cyber Features:
['cyber_incident_count', 'cyber_intensity_score', 'scam_spike_rate', 'cyber_attack_frequency', 'cyber_growth_rate', 'phishing_activity_level', 'misinformation_volume', 'cyber_volatility', 'attack_acceleration', 'incident_peak_flag']

Temporal Features:
['rolling_cyber_mean', 'rolling_cyber_std', 'cyber_spike', 'trend_slope_cyber_activity', 'time_since_last_event', 'moving_average_ratio', 'ema', 'lag_1', 'lag_2', 'time_decay_factor']

Geo Features:
['geo_risk_zone_score', 'high_risk_region_flag', 'incident_density_by_region', 'location_encoded', 'regional_event_count', 'geo_cluster_id']

Risk Features:
['combined_risk_index', 'event_impact_multiplier', 'disaster_cyber_correlation_score', 'anomaly_risk_signal', 'risk_trend_index', 'multi_factor_risk_score', 'hazard_w

## Interpretation of Generated Features

The generated feature set can be grouped into six major categories:

- **Hazard features** capture event severity, intensity, volatility, and hazard-related patterns.
- **Cyber features** capture cyber incident count, growth, spikes, attack activity, and volatility.
- **Temporal features** capture rolling trends, lags, time gaps, and moving averages.
- **Geo features** capture region-level activity, encoding, density, and location-based risk.
- **Combined risk features** combine hazard and cyber signals to produce overall risk indicators.
- **Anomaly features** help identify unusual patterns, threshold breaches, and outlier behaviour.

These features are useful because they convert raw disaster and cyber-related fields into structured AI-ready inputs for future machine learning models.

## Save Mapping File

In [ ]:
mapping_df.to_csv("feature_mapping_ai004.csv", index=False)
print("Feature mapping file saved successfully.")

Feature mapping file saved successfully.


## Download File

In [ ]:
from google.colab import files
files.download("feature_mapping_ai004.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary of Contribution

In this notebook, I reviewed the current feature engineering workflow  and focused on data understanding, feature mapping and validation support. I identified the main raw input fields, mapped them to the engineered features generated by the pipeline and performed basic validation checks on the feature dataset. This work helps document the transformation logic and supports the team in ensuring the engineered features are interpretable, complete and suitable for downstream AI/ML tasks.